In [ ]:
# Pipeline parameters — overridden by job base_parameters when run via DAB.
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "dev")
dbutils.widgets.text("volume_name", "raw_files")
dbutils.widgets.text("bronze_write_mode", "overwrite")
dbutils.widgets.text("overwrite_schema", "true")
dbutils.widgets.text("null_high_severity_pct", "10")
dbutils.widgets.text("dq_sample_limit", "20")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
bronze_write_mode = dbutils.widgets.get("bronze_write_mode")
overwrite_schema = dbutils.widgets.get("overwrite_schema").lower() == "true"
null_high_severity_pct = float(dbutils.widgets.get("null_high_severity_pct"))
dq_sample_limit = int(dbutils.widgets.get("dq_sample_limit"))

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
print(f"catalog={catalog}  schema={schema}  volume_path={volume_path}")
print(f"bronze_write_mode={bronze_write_mode}  overwrite_schema={overwrite_schema}")

## Step 5 - Silver Layer

One table per Bronze source. Each Silver table:
- Drops rows that fail **NULL checks** on required business columns
- Enforces **business rule** filters (date ordering, positive amounts)
- `bronze_risk_zone_lookup` → deduplicates to **one row per postcode** (alphabetically lowest `region_name` retained)
- Renames `ingestion_timestamp` → `bronze_ingestion_timestamp` and adds `silver_ingestion_timestamp` for lineage

In [ ]:
from actuarial_claim_pipeline.silver import build_silver_tables

silver_tables = build_silver_tables(spark, catalog, schema)

for name, df in silver_tables.items():
    print(f"{catalog}.{schema}.{name}: {df.count():,} rows")

In [ ]:
# Silver cyclone events written by build_silver_tables above.
display(spark.table(f"{catalog}.{schema}.silver_cyclone_events").limit(20))

In [ ]:
# Silver premiums written by build_silver_tables above.
display(spark.table(f"{catalog}.{schema}.silver_premium_bordereau").limit(20))

In [ ]:
# Silver risk zones written by build_silver_tables above (one row per postcode).
display(spark.table(f"{catalog}.{schema}.silver_risk_zone_lookup").limit(20))